In [ ]:
from bs4 import BeautifulSoup
import requests
import time
import json
from dataclasses import asdict
from hazm import Normalizer
import re
from dataclasses import dataclass, field
from typing import Optional



In [ ]:
def fetch_document(url, timeout=60):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/149.0.0.0 Safari/537.36"
        )
    }
    while True:
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            response.encoding = response.apparent_encoding
            return response.text
        except Exception as e:
            print("Error happened!")
            print(e)
            print("retrying...")
            time.sleep(2)
            continue

def extract_document_text(html):
    soup = BeautifulSoup(html, "html.parser")
    content = soup.select_one("div.elementor-widget-theme-post-content")
    if content is None:
        raise ValueError("Document content was not found.")
    for tag in content.select("script, style, noscript"):
        tag.decompose()
    for tag in content.find_all(["br", "hr"]):
        tag.replace_with("\n")
    block_tags = [
        "p",
        "div",
        "section",
        "article",
        "li",
        "blockquote",
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
    ]

    for tag in content.find_all(block_tags):
        tag.insert_before("\n")
        tag.insert_after("\n")
    text = content.get_text(separator="", strip=False)
    lines = [line.strip() for line in text.splitlines()]
    cleaned_lines = []
    previous_empty = False
    for line in lines:
        if not line:
            if previous_empty:
                continue
            previous_empty = True
            cleaned_lines.append("")
        else:
            previous_empty = False
            cleaned_lines.append(line)
    return "\n".join(cleaned_lines).strip()


def get_document_text(url):
    html = fetch_document(url)
    try:
        return extract_document_text(html)
    except Exception as e:
        print("No doc here!!!")
        return None

In [15]:
@dataclass
class Segment:
    type: str
    marker: Optional[str] = None
    header: str = " "
    text: str = " "
    children: list = field(default_factory=list)


STRUCTURAL_MARKERS = [
    "جلد",
    "کتاب",
    "بخش",
    "باب",
    "فصل",
    "مبحث",
    "گفتار",
    "قسمت",
]


PROVISION_MARKERS = [
    "ماده",
    "مادۀ",
    "تبصره",
    "تبصرۀ",
]

NUMBERED_CLAUSE_PATTERN = (
    r"(?m)^[ \t]*"
    r"[0-9۰-۹]+"
    r"[ \t]*(?:ـ|-|–|—|\.)"
    r"(?=\s|$)"
)

ALPHABETIC_CLAUSE_PATTERN = (
    r"(?m)^[ \t]*"
    r"[الف-ی]"
    r"[ \t]*\)"
    r"(?=\s|$)"
)

In [ ]:
def get_marker_patterns():
    patterns = []
    for marker in STRUCTURAL_MARKERS:
        pattern = rf"(?m)^[ \t]*{re.escape(marker)}(?=\s|[:\-–—]|$)"
        patterns.append((marker, "structural", pattern))
    for marker in PROVISION_MARKERS:
        pattern = rf"(?m)^[ \t]*{re.escape(marker)}(?=\s|[:\-–—]|$)"
        patterns.append((marker, "provision", pattern))
    patterns.append(("numbered", "provision", NUMBERED_CLAUSE_PATTERN))
    patterns.append(("alphabetic", "provision", ALPHABETIC_CLAUSE_PATTERN))
    return patterns

patterns = get_marker_patterns()


In [ ]:
def find_first_marker(text: str, parent_marker_type=None):
    best_match = None
    best_marker = None
    best_type = None
    for marker, marker_type, pattern in patterns:
        if parent_marker_type == "provision" and marker_type == "structural":
            continue
        match = re.search(pattern, text)
        if match is None:
            continue
        if best_match is None or match.start() < best_match.start():
            best_match = match
            best_marker = marker
            best_type = marker_type
    if best_match is None:
        return None
    return (best_marker, best_type, best_match)

In [ ]:
def find_all_marker_occurrences(text: str, marker: str, marker_type: str):
    if marker == "numbered":
        pattern = NUMBERED_CLAUSE_PATTERN
    elif marker == "alphabetic":
        pattern = ALPHABETIC_CLAUSE_PATTERN
    else:
        pattern = rf"(?m)^[ \t]*{re.escape(marker)}(?=\s|[:\-–—]|$)"
    return list(re.finditer(pattern, text))

In [ ]:
def split_by_marker(text: str, marker: str, marker_type: str):
    matches = find_all_marker_occurrences(text,marker,marker_type)
    if not matches:
        return [text.strip()]
    chunks = []
    prefix = text[:matches[0].start()].strip()
    if prefix:
        chunks.append(prefix)
    for index, match in enumerate(matches):
        start = match.start()
        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(text)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
    return chunks

In [ ]:
def extract_header(chunk: str, marker: str,marker_type: str):
    if marker_type == "structural":
        match = re.match(r"^[ \t]*.*?(?:\n|$)", chunk)
        if match is None:
            return chunk.strip(), ""
        header = match.group(0).strip()
        content = chunk[match.end():].strip()
        return header, content
    if marker == "numbered":
        match = re.match(NUMBERED_CLAUSE_PATTERN, chunk)
        if match is None:
            return chunk.strip(), ""
        header = match.group(0).strip()
        content = chunk[match.end():].strip()
        return header, content
    if marker == "alphabetic":
        match = re.match(ALPHABETIC_CLAUSE_PATTERN, chunk)
        if match is None:
            return chunk.strip(), ""
        header = match.group(0).strip()
        content = chunk[match.end():].strip()
        return header, content
    match = re.match(rf"^[ \t]*{re.escape(marker)}(?:[ \t]+[0-9۰-۹]+)?[ \t]*(?:ـ|-|–|—|:)[ \t]*", chunk)
    if match is not None:
        header = match.group(0).strip()
        content = chunk[match.end():].strip()
        return header, content
    match = re.match(rf"^[ \t]*{re.escape(marker)}.*?(?:\n|$)", chunk)
    if match is None:
        return chunk.strip(), ""
    header = match.group(0).strip()
    content = chunk[match.end():].strip()
    return header, content

In [ ]:
def parse_structure(text: str, parent_marker_type=None):
    text = text.strip()
    if not text:
        return None
    marker_info = find_first_marker(text, parent_marker_type=parent_marker_type)
    if marker_info is None:
        return Segment(type="unscoped", text=text)
    marker, marker_type, _ = marker_info
    chunks = split_by_marker(text,  marker, marker_type)
    if marker_type == "structural":
        root_type = "group"
    else:
        root_type = "provisions"
    root = Segment(type=root_type, marker=marker)
    for chunk in chunks:
        if not find_all_marker_occurrences(chunk, marker, marker_type):
            root.children.append(Segment(type="introduction", text=chunk))
            continue
        header, content = extract_header(chunk, marker, marker_type)
        node_type = "structural" if marker_type == "structural" else "provision"
        node = Segment( type=node_type, marker=marker, header=header)
        if content:
            child = parse_structure(content, parent_marker_type=marker_type)
            if child is not None:
                if child.type == "unscoped":
                    node.text = content
                else:
                    if node_type == "provision":
                        remaining_children = []
                        for child_node in child.children:
                            if not node.text.strip() and child_node.type == "introduction":
                                node.text = child_node.text
                            else:
                                remaining_children.append(child_node)
                        child.children = remaining_children
                    node.children.append(child)
        root.children.append(node)
    return root

In [ ]:
normalizer = Normalizer()

ALLOWED_FIRST_WORDS = {
    "قانون",
    "آیین‌نامه",
    "آییننامه",
    "آیین",
    "تصویب‌نامه",
    "تصویبنامه",
    "مقررات",
    "اساسنامه",
    "نظام‌نامه",
    "نظامنامه",
    "بخشنامه",
    "دستورالعمل",
    "شیوه‌نامه",
    "شیوه",
    "مصوبه",
    "مصوبات",
    "تصمیم‌نامه",
    "تصمیم",
    "لایحه",
    "الحاق",
    "الحاقیه",
    "اصلاح",
    "اصلاحیه",
    "اصلاحات",
    "تمدید",
    "لغو",
    "نسخ",
    "متمم",
    "تکمله",
    "تأسیس",
    "تاسیس",
    "تعیین",
    "تشکیل",
    "ایجاد",
    "واگذاری",
    "انتقال",
    "موافقتنامه",
    "موافقت‌نامه",
    "معاهده",
    "عهدنامه",
    "کنوانسیون",
    "پروتکل",
    "قطعنامه",
    "فرمان",
    "منشور",
    "ضوابط",
    "مقرره",
}


PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)

def normalize_persian_text(text):
    if not text:
        return ""
    text = normalizer.normalize(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def convert_digits(text):
    if not text:
        return ""
    return text.translate(PERSIAN_DIGITS).translate(ARABIC_DIGITS)


def extract_year(text):
    if not text:
        return None
    text = convert_digits(text)
    match = re.search(r"\b(1[0-9]{3}|[0-9]{4})\b", text)
    if not match:
        return None
    return int(match.group(1))


def extract_first_word(title):
    if not title:
        return None

    title = title.strip()
    match = re.match(r"^(\S+)", title)
    if not match:
        return None
    return match.group(1)

def is_foreign(text):
    return "قانون خارجی" in text

def allow_link(title, approval_date, category):
    title = normalize_persian_text(title)
    first_word = extract_first_word(title)
    if not first_word:
        return False
    approval_date = convert_digits(approval_date)
    year = extract_year(approval_date)
    category = normalize_persian_text(category)
    foreign_rule = is_foreign(category)
    if year is None:
        return False
    return first_word in ALLOWED_FIRST_WORDS and year >= 1300 and not foreign_rule

In [23]:
with open("./links/nezamat.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
    for i in range(0 * 10, len(lines), 10):
        print(f"Now on section {i//10}: ")
        current = lines[i:10+i]
        
        link = current[1].replace("url---", "").strip()
        title = current[0].replace("title---", "").strip()
        category = current[6].replace("category---", "").strip()
        approval_date = current[3].replace("approval_date---", "").strip()
        publication_date = current[5].replace("publication_date---", "").strip()
        if not allow_link(title, approval_date, category):
            print(f"section {i//10} skipped ")
            continue
        text = get_document_text(link)
        if text is None:
            continue
        root = parse_structure(text)
        data = {
            "title": title,
            "url": link,
            "category": category,
            "approval_date": approval_date,
            "publication_date": publication_date,
            "structure": asdict(root) if root else None,
        }

        with open(f"./json/{i // 10}.json", "w", encoding="utf-8") as out:
            json.dump(data, out,  ensure_ascii=False, indent=2)
    

Now on section 0: 
section 0 skipped 
Now on section 1: 
section 1 skipped 
Now on section 2: 
section 2 skipped 
Now on section 3: 
section 3 skipped 
Now on section 4: 
section 4 skipped 
Now on section 5: 
section 5 skipped 
Now on section 6: 
section 6 skipped 
Now on section 7: 
section 7 skipped 
Now on section 8: 
section 8 skipped 
Now on section 9: 
Anti-bot challenge detected.
Progress:  | Status: در انتظار تعامل کاربر | URL: https://nezamat.ir/anti-bot-challenge/index.html?redirect=/%d9%85%d8%b5%d9%88%d8%a8%d8%a7%d8%aa-%d8%ac%d9%84%d8%b3%d9%87-%d9%86%d8%ae%d8%b3%d8%aa-%d8%b4%d9%88%d8%b1%d8%a7%db%8c-%d9%85%d8%b1%da%a9%d8%b2%db%8c-%d8%b3%d8%aa%d8%a7%d8%af-%d8%a7%d9%85%d8%b1-%d8%a8/
Progress:  | Status: در انتظار تعامل کاربر | URL: https://nezamat.ir/anti-bot-challenge/index.html?redirect=/%d9%85%d8%b5%d9%88%d8%a8%d8%a7%d8%aa-%d8%ac%d9%84%d8%b3%d9%87-%d9%86%d8%ae%d8%b3%d8%aa-%d8%b4%d9%88%d8%b1%d8%a7%db%8c-%d9%85%d8%b1%da%a9%d8%b2%db%8c-%d8%b3%d8%aa%d8%a7%d8%af-%d8%a7%d9%85%d8%b

KeyboardInterrupt: 